# Practice: describing real data with pandas
*AI from scratch, probability 2* · lesson: https://ihor.sh/ai/p2-describing-data.html

The first thing an ML engineer does with any dataset: look at it. Typical values, spread, outliers, skew, and
which columns move together, on this site's NYC subway outage data and scikit-learn's wine data.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).
On Kaggle, turn on *Internet* in the notebook settings.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Compute mean and median of df.hours.', 'Compute share_below_mean = (df.hours < mean_h).mean().'], 2: ['Compute quartiles q1, q3 using df.hours.quantile([0.25, 0.75]).', 'Compute iqr = q3 - q1.', 'Count outages above Q3 + 1.5 * iqr using (df.hours > q3 + 1.5 * iqr).sum().'], 3: ['Compute skewness of raw hours with df.hours.skew().', 'Compute skewness of log-transformed hours with np.log10(df.hours).skew().'], 4: ['Compute overall Pearson correlation: df.estimated.corr(df.hours).', 'Filter df for hours < 12 and compute correlation on that subset.'], 5: ['pairs is labeled by (name, name); ask it for the label of its maximum.', 'pair = pairs.idxmax()'], 6: ['Standardize wine dataframe: (wine - wine.mean()) / wine.std(ddof=0).']}
_ANSWERS = {1: 'bWVhbl9oLCBtZWRpYW5faCA9IGRmLmhvdXJzLm1lYW4oKSwgZGYuaG91cnMubWVkaWFuKCkKc2hhcmVfYmVsb3dfbWVhbiA9IChkZi5ob3VycyA8IG1lYW5faCkubWVhbigp', 2: 'cTEsIHEzID0gZGYuaG91cnMucXVhbnRpbGUoWzAuMjUsIDAuNzVdKQppcXIgPSBxMyAtIHExCm5fb3V0bGllcnMgPSAoZGYuaG91cnMgPiBxMyArIDEuNSAqIGlxcikuc3VtKCk=', 3: 'c2tld19yYXcgPSBkZi5ob3Vycy5za2V3KCkKc2tld19sb2cgPSBucC5sb2cxMChkZi5ob3Vycykuc2tldygpCmZpZywgYXggPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTAsIDMpKQpkZi5ob3Vycy5oaXN0KGJpbnM9NDAsIGF4PWF4WzBdKTsgYXhbMF0uc2V0X3RpdGxlKCJob3VycyIpCm5wLmxvZzEwKGRmLmhvdXJzKS5oaXN0KGJpbnM9NDAsIGF4PWF4WzFdKTsgYXhbMV0uc2V0X3RpdGxlKCJsb2cxMChob3VycykiKQpwbHQuc2hvdygp', 4: 'cl9hbGwgPSBkZi5lc3RpbWF0ZWQuY29ycihkZi5ob3VycykKc2hvcnQgPSBkZltkZi5ob3VycyA8IDEyXQpyX3Nob3J0ID0gc2hvcnQuZXN0aW1hdGVkLmNvcnIoc2hvcnQuaG91cnMp', 5: 'cGFpciA9IHBhaXJzLmlkeG1heCgp', 6: 'eiA9ICh3aW5lIC0gd2luZS5tZWFuKCkpIC8gd2luZS5zdGQoZGRvZj0wKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/"
data = json.load(urllib.request.urlopen(DATA + "outages.json"))
df = pd.DataFrame(data["rows"], columns=data["columns"])
df.describe()

## 1. Typical values
Set `mean_h` and `median_h` for the `hours` column, and `share_below_mean`: the fraction of outages shorter than
the mean. What does that say about using the mean as "typical" here?

In [ ]:
mean_h = median_h = share_below_mean = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(mean_h, median_h, share_below_mean)

In [ ]:
check(1, lambda: round(mean_h, 2) == 8.25 and median_h == 1.62, 
      "mean_h should round to 8.25 hours and median_h should be 1.62 hours.")
check(1, lambda: share_below_mean > 0.9, 
      "Over 90% of outages should be shorter than the mean due to heavy right skew.")

## 2. Outliers by the 1.5 × IQR rule
A box plot marks a point as an outlier if it's more than 1.5 IQR above Q3 (or below Q1). Using `.quantile([0.25, 0.75])`, set `q1`, `q3`, `iqr`,
and `n_outliers`: how many outages lie above $Q_3 + 1.5\,\text{IQR}$.

In [ ]:
q1 = q3 = iqr = n_outliers = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(q1, q3, iqr, n_outliers)
df.boxplot(column="hours", by="kind", figsize=(6, 3)); plt.yscale("log"); plt.suptitle(""); plt.show()

In [ ]:
check(2, lambda: abs(q1 - 0.95) < 0.01 and abs(q3 - 3.56) < 0.01, 
      "Q1 should be ~0.95 hours and Q3 should be ~3.56 hours.")
Q1, Q3 = df.hours.quantile(0.25), df.hours.quantile(0.75)
check(2, lambda: n_outliers == (df.hours > Q3 + 1.5 * (Q3 - Q1)).sum(), 
      "n_outliers must equal the count of outages where hours > Q3 + 1.5 * IQR.")

## 3. Skew, and the log fix
Set `skew_raw` to the skewness of `hours` (`.skew()`) and `skew_log` to that of `np.log10(hours)`. Then plot both
histograms side by side.

In [ ]:
skew_raw = skew_log = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(skew_raw, skew_log)

In [ ]:
check(3, lambda: skew_raw > 10 and 0.5 < skew_log < 2, 
      "Raw hours should be severely skewed (> 10), while log10(hours) brings skewness between 0.5 and 2.0.")

## 4. A correlation decided by a few points
Set `r_all`: the correlation of `estimated` and `hours` over all outages, and `r_short`: the same over outages
shorter than 12 hours. What does the difference tell you?

In [ ]:
r_all = r_short = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(r_all, r_short)

In [ ]:
check(4, lambda: r_all > 0.98 and r_short < 0.3, 
      "Overall correlation r_all should exceed 0.98, but drop below 0.3 for short outages (< 12h).")

## 5. Which features move together?
scikit-learn's wine data: 178 wines, 13 chemical measurements. `.corr()` gives the correlation of every pair of
columns; the heat map shows it (red: rise together, blue: one rises as the other falls). Strongly correlated
features carry overlapping information: L1 and L2 showed what that does to a linear model.

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True).frame.drop(columns="target")
C = wine.corr()
plt.figure(figsize=(6, 5)); plt.imshow(C, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(13), C.columns, rotation=90); plt.yticks(range(13), C.columns); plt.show()

# keep each pair of different columns once (the part above the diagonal), as a list labeled by the two names
pairs = C.where(np.triu(np.ones(C.shape, dtype=bool), k=1)).stack()
pairs.sort_values().tail(3)

Set `pair` to the label of the most correlated pair: `.idxmax()` gives the label of a Series' biggest value.

In [ ]:
pair = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print(pair)

In [ ]:
check(5, lambda: tuple(sorted(pair)) == ("flavanoids", "total_phenols"), 
      "The most correlated pair of chemical measurements should be ('flavanoids', 'total_phenols').")

## 6. Standardize
Make `z`: every wine column turned into z-scores (subtract the column mean, divide by its standard deviation,
with `ddof=0`). This is the feature scaling that makes gradient descent behave (calculus 3).

In [ ]:
z = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...

In [ ]:
check(6, lambda: np.allclose(z.mean(), 0) and np.allclose(z.std(ddof=0), 1), 
      "Standardized features z must have mean 0 and standard deviation 1 for every column.")
from sklearn.preprocessing import StandardScaler
check(6, lambda: np.allclose(z.values, StandardScaler().fit_transform(wine)), 
      "Manual standardization must match sklearn's StandardScaler().fit_transform(wine).")

## Next
Back to the path: https://ihor.sh/ai/ · next: linear algebra 2, eigenvectors and PCA.